# Silver 2: retry the failed parses

Task `silver_retry` of the job `ingest_financebench`. Takes the parts whose parse failed in a **retryable** way (rate limit, internal error) and have attempts left, and parses them again with the **same function** as the first parse (`parse_docs`). There is no special repair code: a part that now parses completely goes to `silver_parsed`, and one that fails again gets another row in `silver_log` with the next attempt number.

Every retryable PDF is retried in one run. To go easy on a limited quota (the Databricks free tier):

| Guard | What it does |
| --- | --- |
| Cheapest first | The shortest parts go first |
| One part at a time, with a pause | `retry_pause_seconds`, so the rate limit is not hit again |
| Stop when the service is limiting | Two failures in a row stop the retries for this run |
| Attempt cap | After `max_attempts` a part is left alone. It stays in `silver_failures` |
| Never retried | Failures that cannot succeed (a failed conversion) |

A retry parses the whole **part** again, at most `part_pages` pages. That is what the split is for: retrying a 200-page PDF costs 200 parse pages, retrying one of its parts costs 50.

In [ ]:
import time

from tables import Names, create_tables
from parsing import parse_docs

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "finance_bench_rag")
dbutils.widgets.text("max_attempts", "3")
dbutils.widgets.text("retry_pause_seconds", "10")

n = Names(dbutils.widgets.get("catalog"), dbutils.widgets.get("schema"))
max_attempts = int(dbutils.widgets.get("max_attempts"))
retry_pause_seconds = int(dbutils.widgets.get("retry_pause_seconds"))

create_tables(spark, n)

In [ ]:
candidates = spark.sql(f"""
SELECT path, part_no, pdf_pages FROM {n.failures}
WHERE stage = 'parse' AND retryable AND attempts < {max_attempts}
ORDER BY pdf_pages
""").collect()
print(f"{len(candidates)} parts have a retryable parse failure.")

spent, in_a_row = 0, 0
for c in candidates:
    if in_a_row >= 2:  # the service keeps failing: stop, run again later
        break
    part = spark.table(n.parts).filter(f"path = '{c.path}' AND part_no = {c.part_no}")
    good, bad = parse_docs(spark, n, part)
    spent += c.pdf_pages
    in_a_row = in_a_row + 1 if bad else 0
    print(f"{c.path.split('/')[-1]} part {c.part_no}: {'parsed' if good else 'failed again'} ({c.pdf_pages} pages)")
    time.sleep(retry_pause_seconds)
print(f"Retry pages spent: {spent}." + (" Stopped: the service keeps failing." if in_a_row >= 2 else ""))